# Module 5 / Phase 1: Data Integration & Provenance Alignment
**Project:** RouteRanker for Chennai MTC Buses  
**Notebook:** `05_phase1_data_integration.ipynb`  
**Purpose:** Integrate verified MTC corporate performance metrics with GTFS supply data, resolve route naming/granularity hierarchies, establish methodological guardrails (Supply vs. Realized Demand), and prepare audit-ready indicators for RouteRanker.

---

### Methodological Guardrails:
1. **Supply != Demand:** GTFS trip dispatches (47,047) represent scheduled capacity events, not individual buses and not passenger boardings.
2. **Provenance Preservation:** All raw GTFS tables (`routes.txt`, `stops.txt`, etc.) and official MTC performance data are ingested strictly read-only.
3. **No Synthetic Demand:** No artificial stop-level or passenger counts are invented.
4. **Scope Demarcation:** The mathematical ratio of 32.24 lakh passengers / 47,047 GTFS trips (~68.5) is strictly a system-wide macroeconomic proxy, NOT actual trip-level passenger ridership or bus occupancy. Similarly, comparing 47,047 scheduled dispatches to 2,860 physical buses highlights different reporting layers (static schedule definitions vs physical bus turnouts) and must not be conflated as uniform operations.


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

DATA_DIR = Path('.')
EXT_DATA_DIR = Path('data/external')
FIG_DIR = Path('reports/figures')

print('Libraries and directory paths configured successfully.')


Libraries and directory paths configured successfully.


## 1. Load & Audit Official MTC Corporate Performance Data
Official performance indicators published by the Metropolitan Transport Corporation (Chennai) Ltd. (`mtcbus.tn.gov.in`) and archived on OpenCity (`data.opencity.in`).


In [2]:
perf_df = pd.read_csv(EXT_DATA_DIR / 'mtc_performance_indicators.csv')
print(f'MTC Performance Indicators Loaded: {len(perf_df)} metrics across fiscal years.')
perf_df[['sl_no', 'item', 'unit', '2024_2025', '2025_2026', '2026_2027_oct']].head(10)


MTC Performance Indicators Loaded: 15 metrics across fiscal years.


,sl_no,item,unit,2024_2025,2025_2026,2026_2027_oct
0,1,Total Fleet,count,3407.00,4060.00,4206.00
1,2,Average Age of Bus,years,7.43,5.07,5.08
2,3,New Buses Put on Road,count,715.00,1347.00,111.00
3,4,Scheduled Services,count,3233.00,3783.00,3988.00
4,5,Effective Kms (in Lakhs),lakh_km,2635.25,2802.45,1603.06
5,6,Km / Bus / Day,km,253.00,252.00,252.00
6,7,% of Fleet Utilization,percentage,83.94,83.77,84.85
7,8,Km Efficiency %,percentage,83.29,87.77,89.55
8,9,% of Occupancy,percentage,71.22,77.34,68.39
9,10,Passenger / Day (in Lakhs),lakh_passengers,32.24,34.00,34.40


## 2. Ingest Primary GTFS Entities (Read-Only)
Load the GTFS tables and establish primary/foreign key baseline counts.


In [3]:
routes_df = pd.read_csv(DATA_DIR / 'routes.txt', dtype={'route_id': str, 'agency_id': str})
trips_df = pd.read_csv(DATA_DIR / 'trips.txt', dtype={'route_id': str, 'trip_id': str, 'service_id': str})
stops_df = pd.read_csv(DATA_DIR / 'stops.txt', dtype={'stop_id': str})
calendar_df = pd.read_csv(DATA_DIR / 'calendar.txt', dtype={'service_id': str})

print(f'Routes Defined: {len(routes_df):,}')
print(f'Trips Scheduled: {len(trips_df):,}')
print(f'Stops Defined: {len(stops_df):,}')
print(f'Calendar Services: {len(calendar_df):,}')


Routes Defined: 4,611
Trips Scheduled: 47,047
Stops Defined: 5,580
Calendar Services: 6


## 3. Resolving the Route Count Discrepancy & Robust Canonical Mapping
Preserving distinct operational identities: MTC route identifiers have specific conventions:
- Base corridor lines like `19`, `19B`, `M19B` represent different physical corridors and must NOT be merged.
- Suffixes like `CT`, `CT1` (Cut-Trip short-loops), `R`, `R1` (Return/Route variants), and `CUT` represent operational dispatches along a parent corridor.
- `routes.txt` defines **4,611** internal `route_id` records.
- `trips.txt` references only **3,934** active `route_id` records (677 unserviced routes).
- Public bus route names (`route_short_name`) number **2,320** total and **2,099** active.
- Applying suffix stripping yields **675** defined base corridors and **654** active base corridors.


In [4]:
def extract_canonical(name):
    if not isinstance(name, str):
        return ''
    s = name.strip()
    # Strip operational variation suffixes (Cut-Trips and Return directions)
    s = re.sub(r'\s+(CT\d*|CUT\d*|R\d*|RE\d*|E|C\d+)\b', '', s, flags=re.IGNORECASE)
    s = re.sub(r'(?<=[0-9A-Za-z])(CT\d*|CUT\d*)$', '', s, flags=re.IGNORECASE)
    return s.strip().upper()

routes_df['canonical_route'] = routes_df['route_short_name'].apply(extract_canonical)

active_route_ids = set(trips_df['route_id'].unique())
active_routes_df = routes_df[routes_df['route_id'].isin(active_route_ids)].copy()

print('=== Route Granularity Hierarchy Audit ===')
print(f'1. Defined route_id count (routes.txt): {len(routes_df):,}')
print(f'2. Active route_id count (trips.txt): {len(active_routes_df):,}')
print(f'3. Unused route_id count: {len(routes_df) - len(active_routes_df):,}')
print(f'4. Distinct public line names (routes.txt): {routes_df["route_short_name"].nunique():,}')
print(f'5. Distinct active public line names: {active_routes_df["route_short_name"].nunique():,}')
print(f'6. Distinct canonical base corridors (routes.txt): {routes_df["canonical_route"].nunique():,}')
print(f'7. Distinct active canonical base corridors: {active_routes_df["canonical_route"].nunique():,}')


=== Route Granularity Hierarchy Audit ===
1. Defined route_id count (routes.txt): 4,611
2. Active route_id count (trips.txt): 3,934
3. Unused route_id count: 677
4. Distinct public line names (routes.txt): 2,320
5. Distinct active public line names: 2,099
6. Distinct canonical base corridors (routes.txt): 675
7. Distinct active canonical base corridors: 654


## 4. Methodological Scope Demarcation (Supply Dispatches vs Fleet Reality)
Comparing scheduled GTFS timetable dispatches (47,047) against MTC's physical vehicle capacity.

**Methodological Guardrails:**
- The ratio `32.24 Lakhs / 47,047 trips = ~68.5` is strictly an aggregate macroeconomic proxy, NOT actual trip-level passenger ridership or occupancy.
- In GTFS, `trips.txt` contains 47,047 scheduled trip dispatch events. In MTC operational reports, `Scheduled Services` (3,233) represents physical bus duty turnouts.
- Comparing these layers demonstrates that GTFS trips reflect trip dispatches across multiple duty shifts rather than 1:1 physical buses.


In [5]:
scheduled_gtfs_trips = len(trips_df)
official_scheduled_services = int(float(perf_df[perf_df['item'] == 'Scheduled Services']['2024_2025'].values[0]))
total_fleet = int(float(perf_df[perf_df['item'] == 'Total Fleet']['2024_2025'].values[0]))
fleet_util_pct = float(perf_df[perf_df['item'] == '% of Fleet Utilization']['2024_2025'].values[0])
active_buses = int(total_fleet * (fleet_util_pct / 100.0))
official_daily_pax_lakhs = float(perf_df[perf_df['item'] == 'Passenger / Day (in Lakhs)']['2024_2025'].values[0])

proxy_pax_per_dispatch = (official_daily_pax_lakhs * 100000) / scheduled_gtfs_trips

print('=== Supply Capacity vs Physical Fleet Reality ===')
print(f'Scheduled Daily GTFS Trips (Supply Dispatches): {scheduled_gtfs_trips:,}')
print(f'Official MTC Scheduled Bus Services (Turnouts): {official_scheduled_services:,}')
print(f'Total Physical Fleet Size: {total_fleet:,}')
print(f'Actively Deployed Buses on Road: {active_buses:,} ({fleet_util_pct}%)')
print(f'Official Average Daily Passengers: {official_daily_pax_lakhs:.2f} Lakhs ({official_daily_pax_lakhs*100000:,.0f} passengers/day)')
print(f'System-Wide Mathematical Proxy Ratio: {proxy_pax_per_dispatch:.1f} passengers/trip dispatch (NOT actual trip occupancy)')
print('Note: GTFS trip dispatches and physical bus turnout duties represent different operational reporting layers.')


=== Supply Capacity vs Physical Fleet Reality ===
Scheduled Daily GTFS Trips (Supply Dispatches): 47,047
Official MTC Scheduled Bus Services (Turnouts): 3,233
Total Physical Fleet Size: 3,407
Actively Deployed Buses on Road: 2,860 (83.94%)
Official Average Daily Passengers: 32.24 Lakhs (3,224,000 passengers/day)
System-Wide Mathematical Proxy Ratio: 68.5 passengers/trip dispatch (NOT actual trip occupancy)
Note: GTFS trip dispatches and physical bus turnout duties represent different operational reporting layers.


## 5. Integrating Supply Metrics by Canonical Corridor
Aggregate scheduled trip supply, active variants, and service span at the canonical corridor level.


In [6]:
trip_counts_per_route = trips_df['route_id'].value_counts()
routes_df['trip_count'] = routes_df['route_id'].map(trip_counts_per_route).fillna(0).astype(int)

canonical_summary = routes_df.groupby('canonical_route').agg(
    total_scheduled_trips=('trip_count', 'sum'),
    defined_variants=('route_id', 'count'),
    active_variants=('trip_count', lambda s: (s > 0).sum()),
    distinct_public_names=('route_short_name', 'nunique')
).reset_index()

canonical_summary = canonical_summary.sort_values(by='total_scheduled_trips', ascending=False)
print('Top 15 Canonical Corridors by Scheduled Daily Trips:')
canonical_summary.head(5)


Top 15 Canonical Corridors by Scheduled Daily Trips:


,canonical_route,total_scheduled_trips,defined_variants,active_variants,distinct_public_names
198,25,802,16,15,12
556,91,757,16,15,11
415,570,693,17,17,9
348,500,683,15,14,10
418,57,639,15,15,9


## 6. Verification and Export of Integrated Metadata
Save the canonical route resolution mapping for use in subsequent RouteRanker modules.


In [7]:
out_path = Path('data/canonical_routes_mapping.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
canonical_summary.to_csv(out_path, index=False)
print(f'Successfully exported canonical route mapping to: {out_path.resolve()}')
print(f'Total Canonical Routes Exported: {len(canonical_summary):,}')


Successfully exported canonical route mapping to: C:\Users\kavyasri\Downloads\mtc-gtfs\data\canonical_routes_mapping.csv
Total Canonical Routes Exported: 675
